In [70]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import  pandas as pd 
import json 
import os
from glob import glob
import seaborn as sns 
import numpy as np 
import re
import tikzplotly
import plotly.express as px
from IPython.display import display
from PIL import Image
import matplotlib as mpl
import matplotlib.pyplot as plt 
import plotly 
import plotly.graph_objects as go

from IPython.display import IFrame

from utils.benchmark import * 

In [71]:
notebook_name="0-modubft-spread-over-azs"
os.makedirs(f"outputs/{notebook_name}", exist_ok=True)
 

In [72]:
# folder="../../aws/benchmark/out/modubft/full_spread_modubft_with_bytes_sent/" 
folder = "../../aws/benchmark/out/modubft/20251022-011614/"
os.listdir(folder)

['20251022011946-cb1-v512',
 '20251022012021-cb1-v4096',
 '20251022012544-cb1-v512',
 '20251022012621-cb1-v4096',
 '20251022013346-cb1-v512',
 '20251022013433-cb1-v4096',
 '20251022014440-cb1-v512',
 '20251022014533-cb1-v4096',
 '20251022015656-cb1-v512',
 '20251022015755-cb1-v4096',
 'modubft_peers_12.txt',
 'modubft_peers_18.txt',
 'modubft_peers_21.txt',
 'modubft_peers_3.txt',
 'modubft_peers_6.txt']

In [73]:
benchmarks = glob(f"{folder}*/")
benchmarks

['../../aws/benchmark/out/modubft/20251022-011614/20251022011946-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251022-011614/20251022012021-cb1-v4096/',
 '../../aws/benchmark/out/modubft/20251022-011614/20251022012544-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251022-011614/20251022012621-cb1-v4096/',
 '../../aws/benchmark/out/modubft/20251022-011614/20251022013346-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251022-011614/20251022013433-cb1-v4096/',
 '../../aws/benchmark/out/modubft/20251022-011614/20251022014440-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251022-011614/20251022014533-cb1-v4096/',
 '../../aws/benchmark/out/modubft/20251022-011614/20251022015656-cb1-v512/',
 '../../aws/benchmark/out/modubft/20251022-011614/20251022015755-cb1-v4096/']

In [74]:


throughputs = process_throughput_benchmarks(benchmarks)

In [75]:

throughputs
# Usage:


tikz_plot = TikzPlotGenerator(
    xs=extract_groups_as_lists(throughputs,"num_peers","vallen"),
    ys=extract_groups_as_lists(throughputs,"throughput","vallen"),
    cat=extract_unique_categories(throughputs,"vallen") ,
    xlabel="Number of Peers",
    ylabel="Throughput (ops/sec)",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-throughput.tex"
)
tikz_plot.save()
tikz_plot.compile(output_dir=f"outputs/{notebook_name}/")


[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./outputs/0-modubft-spread-over-azs/modubft-spread-over-azs-throughput.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/s

0

In [76]:


processed_cpu_usage = process_cpu_usage(benchmarks)

role2_processed_cpu_usage = processed_cpu_usage.query("role == 2")

tikz_plot_cpu = TikzPlotGenerator(
    xs=extract_groups_as_lists(role2_processed_cpu_usage,"num_peers","vallen"),
    ys=extract_groups_as_lists(role2_processed_cpu_usage,"cpu_usage","vallen"),
    cat=extract_unique_categories(role2_processed_cpu_usage,"vallen") ,
    xlabel="Number of Peers",
    ylabel="CPU Usage",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-cpu-role2.tex"
)
tikz_plot_cpu.save()
tikz_plot_cpu.compile(output_dir=f"outputs/{notebook_name}/")


role0_processed_cpu_usage = processed_cpu_usage.query("role == 0")
tikz_plot_cpu = TikzPlotGenerator(
    xs=extract_groups_as_lists(role0_processed_cpu_usage,"num_peers","vallen"),
    ys=extract_groups_as_lists(role0_processed_cpu_usage,"cpu_usage","vallen"),
    cat=extract_unique_categories(role0_processed_cpu_usage,"vallen") ,
    xlabel="Number of Peers",
    ylabel="CPU Usage",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-cpu-role0.tex"
)
tikz_plot_cpu.save()
tikz_plot_cpu.compile(output_dir=f"outputs/{notebook_name}/")


concat = [processed_cpu_usage.query("role == 1").groupby("benchmark").max().reset_index().assign(agg="max"),
processed_cpu_usage.query("role == 1").groupby("benchmark").min().reset_index().assign(agg="min")]
concat = pd.concat(concat)
tikz_plot_cpu = TikzPlotGenerator(
    xs=extract_groups_as_lists(concat,"num_peers",["vallen","agg"]),
    ys=extract_groups_as_lists(concat,"cpu_usage",["vallen","agg"]),
    cat=extract_unique_categories(concat,["vallen","agg"]) ,
    xlabel="Number of Peers",
    ylabel="CPU Usage",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-cpu-role1.tex"
  
)
tikz_plot_cpu.save()
tikz_plot_cpu.compile(output_dir=f"outputs/{notebook_name}/")


[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./outputs/0-modubft-spread-over-azs/modubft-spread-over-azs-cpu-role2.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/st

0

In [77]:

preprepare_messages = throughputs["throughput"]*throughputs["num_peers"]
prepare_messages = throughputs["throughput"] *throughputs["num_peers"] * throughputs["num_peers"]
commit_messages = throughputs["throughput"] *(throughputs["num_peers"]-1) * throughputs["num_peers"]
client_responses = throughputs["throughput"]* throughputs["num_peers"]

total_messages = preprepare_messages + prepare_messages + commit_messages + client_responses

throughputs["preprepare_messages"] = preprepare_messages
throughputs["prepare_messages"] = prepare_messages
throughputs["commit_messages"] = commit_messages
throughputs["client_responses"] = client_responses
throughputs["total_messages"] = total_messages/10

# fig = go.Figure() 

# for data in throughputs.groupby("vallen"): 
#     vallen, df = data
#     fig.add_trace(go.Scatter(
#         x=df["num_peers"],
#         y=df["total_messages"],
#         mode="lines+markers",
#         name=f"Vallen={vallen}"
#     ))
# fig.update_layout(
#     title="Total Messages Sent vs Number of Peers",
#     xaxis_title="Number of Peers",
#     yaxis_title="Total Messages Sent (messages/sec)"
# )
# fig.show()
tikz_plot_messages = TikzPlotGenerator(
    xs=extract_groups_as_lists(throughputs,"num_peers","vallen"),
    ys=extract_groups_as_lists(throughputs,"total_messages","vallen"),
    cat=extract_unique_categories(throughputs,"vallen") ,
    xlabel="Number of Peers",
    ylabel="Total Messages Sent (messages/sec)",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-all-messages.tex"
)
tikz_plot_messages.save()
tikz_plot_messages.compile(output_dir=f"outputs/{notebook_name}/")


[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./outputs/0-modubft-spread-over-azs/modubft-spread-over-azs-all-messages.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex

0

In [78]:

preprepare_messages = throughputs["throughput"]*throughputs["num_peers"]
prepare_messages = throughputs["throughput"] *throughputs["num_peers"] 
commit_messages = throughputs["throughput"] *(throughputs["num_peers"]-1) 
client_responses = throughputs["throughput"]

total_messages = preprepare_messages + prepare_messages + commit_messages + client_responses

throughputs["preprepare_messages"] = preprepare_messages
throughputs["prepare_messages"] = prepare_messages
throughputs["commit_messages"] = commit_messages
throughputs["client_responses"] = client_responses
throughputs["total_messages"] = total_messages/10

# fig = go.Figure() 

# for data in throughputs.groupby("vallen"): 
#     vallen, df = data
#     fig.add_trace(go.Scatter(
#         x=df["num_peers"],
#         y=df["total_messages"],
#         mode="lines+markers",
#         name=f"Vallen={vallen}"
#     ))
# fig.update_layout(
#     title="Total Messages Sent vs Number of Peers",
#     xaxis_title="Number of Peers",
#     yaxis_title="Total Messages Sent (messages/sec)"
# )
# fig.show()
tikz_plot_messages = TikzPlotGenerator(
    xs=extract_groups_as_lists(throughputs,"num_peers","vallen"),
    ys=extract_groups_as_lists(throughputs,"total_messages","vallen"),
    cat=extract_unique_categories(throughputs,"vallen") ,
    xlabel="Number of Peers",
    ylabel="Total Messages Sent (messages/sec)",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-messages.tex"
)
tikz_plot_messages.save()
tikz_plot_messages.compile(output_dir=f"outputs/{notebook_name}/")



[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./outputs/0-modubft-spread-over-azs/modubft-spread-over-azs-messages.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/sta

0

In [79]:

bytes_sent = process_bytes_sent(benchmarks)
bytes_sent

,benchmark,num_peers,vallen,runtime,typ,bytes_sent,role
0,../../aws/benchmark/out/modubft/20251022-01161...,3,512,None,ClientResponse,3.695133e+07,2
1,../../aws/benchmark/out/modubft/20251022-01161...,3,512,None,Checkpoint,2.549280e+05,2
2,../../aws/benchmark/out/modubft/20251022-01161...,3,512,None,Commit,1.053744e+08,2
3,../../aws/benchmark/out/modubft/20251022-01161...,3,512,None,PrePrepare,1.005887e+09,2
4,../../aws/benchmark/out/modubft/20251022-01161...,3,512,None,Prepare,1.276002e+08,2
...,...,...,...,...,...,...,...
695,../../aws/benchmark/out/modubft/20251022-01161...,21,4096,None,ClientResponse,2.973701e+05,1
696,../../aws/benchmark/out/modubft/20251022-01161...,21,4096,None,Checkpoint,1.694280e+04,1
697,../../aws/benchmark/out/modubft/20251022-01161...,21,4096,None,Commit,8.356390e+06,1
698,../../aws/benchmark/out/modubft/20251022-01161...,21,4096,None,PrePrepare,0.000000e+00,1


In [80]:
bytes_sent_in_gbps = (bytes_sent.groupby(["num_peers","vallen","role"])[["bytes_sent"]].sum() * 8 * 10**(-9) * 15**(-1)).reset_index()

bytes_sent_role2 = bytes_sent_in_gbps.query("role == 2")
tikz_plot_messages = TikzPlotGenerator(
    xs=extract_groups_as_lists(bytes_sent_role2,"num_peers","vallen"),
    ys=extract_groups_as_lists(bytes_sent_role2,"bytes_sent","vallen"),
    cat=extract_unique_categories(bytes_sent_role2,"vallen") ,
    xlabel="Number of Peers",
    ylabel="Bytes Sent (Gbps)",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role2.tex"
)
tikz_plot_messages.save()
tikz_plot_messages.compile(output_dir=f"outputs/{notebook_name}/")

bytes_sent_role1 = bytes_sent_in_gbps.query("role == 1")
tikz_plot_messages = TikzPlotGenerator(
    xs=extract_groups_as_lists(bytes_sent_role1,"num_peers","vallen"),
    ys=extract_groups_as_lists(bytes_sent_role1,"bytes_sent","vallen"),
    cat=extract_unique_categories(bytes_sent_role1,"vallen") ,
    xlabel="Number of Peers",
    ylabel="Bytes Sent (Gbps)",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role1.tex"
)
tikz_plot_messages.save()
tikz_plot_messages.compile(output_dir=f"outputs/{notebook_name}/")



[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/0-modubft-spread-over-azs/modubft-spread-over-azs-bytes-sent-role2.t
ex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex

0

In [81]:


# Example usage:
role_df = get_role_df(bytes_sent, role=2)
role_df_complete = complete_multiindex(role_df, ['num_peers', 'typ', 'vallen'])
role_df_complete = add_combined_column(role_df_complete, ['num_peers', 'vallen'], 'num_peers_vallen')
role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')

plot_tikz(
    role_df_complete,
    x_col="num_peers_vallen",
    y_col="bytes_sent_percent",
    cat_col="typ",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role2.tex",
    xlabel="(Number of Peers, Vallen)",
    ylabel="Bytes Sent Percent",
    bar=True,
    stack=True,
    symbolic_x=True,
    sort_x_key=lambda x: eval(x),
    output_dir=f"outputs/{notebook_name}/"
    
)
role_df = get_role_df(bytes_sent, role=1)
role_df_complete = complete_multiindex(role_df, ['num_peers', 'typ', 'vallen'])
role_df_complete = add_combined_column(role_df_complete, ['num_peers', 'vallen'], 'num_peers_vallen')
role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')
plot_tikz(
    role_df_complete,
    x_col="num_peers_vallen",
    y_col="bytes_sent_percent",
    cat_col="typ",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-bytes-sent-role1.tex",
    xlabel="(Number of Peers, Vallen)",
    ylabel="Bytes Sent Percent",
    bar=True,
    stack=True,
    symbolic_x=True,
    sort_x_key=lambda x: eval(x),
    output_dir=f"outputs/{notebook_name}/"
)


[np.str_('(3, 512)'), np.str_('(3, 4096)'), np.str_('(6, 512)'), np.str_('(6, 4096)'), np.str_('(12, 512)'), np.str_('(12, 4096)'), np.str_('(18, 512)'), np.str_('(18, 4096)'), np.str_('(21, 512)'), np.str_('(21, 4096)')]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/0-modubft-spread-over-azs/modubft-spread-over-azs-bytes-sent-role2.t
ex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share

0

In [82]:
res = {}
array_length = 1000
for benchmark in benchmarks: 

    received = np.zeros(array_length)
    response = np.zeros(array_length)
    for log in glob(f"{benchmark}/*.log"): 
        
        with open(log, 'r') as f:
            lines = [line for line in f.read().splitlines() if "client request" in line]

        if len(lines) != 2*array_length : 
            continue
        if lines : 
           
            for line in lines : 
                nanoseconds = line.split(" ")[-1]
                if "Received" in line: 
                    rc = int(line.split(" ")[3])
                    # print(line)
                    received[rc] = int(nanoseconds)
                    rc += 1
                elif "Response" in line: 
                    rp = int(line.split(" ")[3])
                    response[rp] = int(nanoseconds)
                    rp += 1
                else: 
                    raise ValueError("Unexpected line")
            
            latency = response - received
    latency_ms = latency * 1e-6
    max_latency = np.max(latency_ms)
    avg_latency = np.median(latency_ms)
    min_latency = np.min(latency_ms)
    throughputs.loc[throughputs['benchmark'] == benchmark, 'latency_ms'] = avg_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'max_latency_ms'] = max_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'min_latency_ms'] = min_latency
    
    if min_latency < 0 : 
        raise ValueError("Negative latency detected")
    print(f"Benchmark: {benchmark}, Max Latency: {max_latency}, Avg Latency: {avg_latency}, Min Latency: {min_latency}")
        

Benchmark: ../../aws/benchmark/out/modubft/20251022-011614/20251022011946-cb1-v512/, Max Latency: 25.809151999999997, Avg Latency: 20.649088, Min Latency: 5.217536
Benchmark: ../../aws/benchmark/out/modubft/20251022-011614/20251022012021-cb1-v4096/, Max Latency: 58.348544, Avg Latency: 42.667263999999996, Min Latency: 12.09088
Benchmark: ../../aws/benchmark/out/modubft/20251022-011614/20251022012544-cb1-v512/, Max Latency: 49.511424, Avg Latency: 45.08057599999999, Min Latency: 8.660224
Benchmark: ../../aws/benchmark/out/modubft/20251022-011614/20251022012621-cb1-v4096/, Max Latency: 81.686528, Avg Latency: 74.088448, Min Latency: 38.021119999999996
Benchmark: ../../aws/benchmark/out/modubft/20251022-011614/20251022013346-cb1-v512/, Max Latency: 82.684928, Avg Latency: 77.594624, Min Latency: 5.864704
Benchmark: ../../aws/benchmark/out/modubft/20251022-011614/20251022013433-cb1-v4096/, Max Latency: 142.97548799999998, Avg Latency: 114.941056, Min Latency: 11.480063999999999
Benchmark: 

In [83]:
plot_tikz(
    throughputs,
    y_col="latency_ms",
    x_col="num_peers",
    cat_col="vallen",
    filename=f"outputs/{notebook_name}/modubft-spread-over-azs-latency.tex",
    xlabel="Number of Peers",
    ylabel="Latency (ms)",
    output_dir=f"outputs/{notebook_name}/"
)

[np.int64(3), np.int64(6), np.int64(12), np.int64(18), np.int64(21)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./outputs/0-modubft-spread-over-azs/modubft-spread-over-azs-latency.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/stan

0